# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration of the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

FAIR<sup>2</sup> dataset: Mass spectrometry protein data and related annotations from human mast cell extracellular vesicles.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) available at the following URL:

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This loads the machine-actionable data package and allows us to examine the available record sets and schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (FAIR^2 dataset)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Loaded Dataset: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Explore all available record sets, field `@id`s, and column `@id`s as defined in the Croissant schema. All references use the unique `@id` field attached to each entity. This helps ensure clarity if referencing multiple data resources.

In [ ]:
# List record sets and their fields (all by @id)

print("Record sets in this Croissant dataset:")
for rset in metadata.record_sets:
    print(f"  RecordSet @id: {rset.id}  |  Name: {rset.name}")
    # List fields for each record set
    for field in rset.fields:
        print(f"    Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

#### Example: Print a few records from a chosen record set
All further processing will refer to record sets and fields strictly by their `@id`.

In [ ]:
# Choose a record set by @id. Identify a tabular main record set. Print the first few records.
# For this dataset, the main protein record set is: 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/recordset_proteins'
main_recordset_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/recordset_proteins'

print(f"\nExample records from RecordSet @id: {main_recordset_id}\n---")
for i, record in enumerate(dataset.records(record_set=main_recordset_id)):
    print(record)
    if i >= 2:  # Print only the first three records
        break

## 3. Data Extraction
We now extract all rows from the identified record set into a pandas DataFrame. All references to record sets and fields are by `@id`.

Typically, you can find available record sets in section 2 above, and their fields using the `@id` for reliable reference.

In [ ]:
# Extract all tabular record sets into pandas DataFrames, keyed by record set @id
tabular_recordset_ids = [rset.id for rset in metadata.record_sets]
dataframes = {}

for rec_id in tabular_recordset_ids:
    data = list(dataset.records(record_set=rec_id))
    if len(data) > 0:
        df = pd.DataFrame(data)
        dataframes[rec_id] = df

# List all available DataFrames (by RecordSet @id)
print("\nDataFrames loaded (by RecordSet @id):")
for key in dataframes:
    print(f"  - {key}: shape={dataframes[key].shape}")

# Show available columns for the main record set
df_main = dataframes[main_recordset_id]
print("\nColumns in RecordSet @id {0}:".format(main_recordset_id))
print(df_main.columns.tolist())
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Let's apply basic analyses:
- Filter proteins by coverage percentage or peptide count (by their `@id`s)
- Normalize a numeric column (e.g., molecular weight)
- Group by a category (e.g., protein evidence or sequence status) using field `@id`s

**All references use the field or column `@id` as listed above.**

In [ ]:
# Select field @id for numeric analysis (e.g., % coverage)
coverage_field_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/field_coverage_percent'
mw_field_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/field_molecular_weight_kda'
category_field_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/field_description'

df = dataframes[main_recordset_id]

# Filter: proteins with sequence coverage > 5%
threshold = 5
filtered_df = df[df[coverage_field_id] > threshold]
print(f"Filtered records where {coverage_field_id} > {threshold}:")
print(filtered_df[[coverage_field_id, mw_field_id, category_field_id]].head())

# Normalize molecular weight
norm_field = f"{mw_field_id}_normalized"
filtered_df[norm_field] = (filtered_df[mw_field_id] - filtered_df[mw_field_id].mean()) / filtered_df[mw_field_id].std()
print(f"\n{mw_field_id} normalized:")
print(filtered_df[[mw_field_id, norm_field]].head())

# Optionally group by annotation or description field, and show mean coverage and MW
if category_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(category_field_id)[[coverage_field_id, mw_field_id]].mean().sort_values(coverage_field_id, ascending=False)
    print(f"\nGrouped mean {coverage_field_id} and {mw_field_id} by {category_field_id} (top 5):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of protein molecular weights and coverage. If notebook is run as Colab or in Jupyter, plots will appear inline.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[mw_field_id].dropna(), bins=30, kde=True)
plt.xlabel('Molecular Weight (kDa)')
plt.title('Protein Molecular Weight Distribution')
plt.show()

plt.figure(figsize=(8,4))
sns.histplot(df[coverage_field_id].dropna(), bins=30, kde=True)
plt.xlabel('Coverage (%)')
plt.title('Protein Coverage Percent Distribution')
plt.show()

## 6. Conclusion
- This notebook demonstrated how to use the `mlcroissant` library to programmatically explore and process FAIR<sup>2</sup> protein dataset with all schema entities referenced strictly via their `@id`.
- The dataset describes protein-level data from mass spectrometry analysis of human mast cell extracellular vesicles.
- You can now proceed to perform specialized downstream analysis, such as comparing protein abundance or investigating sequence annotations using the clear, stable `@id`s found in the Croissant schema.

**Further reading:**
- [Croissant MLCommons standard](https://mlcommons.org/croissant/)
- [mlcroissant Python API usage](https://mlcommons.github.io/croissant/python/)
